In [ ]:
import pandas as pd
import polars as pl
import os
import glob
import gc
from tqdm import tqdm

# === 1. 环境清理与路径 ===
output_path = './Dataset/'
# 清理之前可能产生的失败文件
for f in glob.glob(os.path.join(output_path, "*.parquet")):
    try:
        os.remove(f)
    except:
        pass

def find_dataset_path(folder_name):
    for root, dirs, files in os.walk('/kaggle/input/datasets/lan720'):
        if folder_name.lower() in root.lower():
            return root
    return f'/kaggle/input/datasets/lan720/{folder_name}'

dataset_path = "E:\AliRec\mobileresdataset"
print(f"📂 数据集路径: {dataset_path}")

# === 2. 核心保存逻辑（修改版：强制指定 Schema） ===

def _save_chunk(data, schema, prefix, idx):
    """
    辅助函数：将列表保存为 parquet，强制使用指定 Schema
    """
    filename = f"{prefix}_{idx:03d}.parquet"
    save_path = os.path.join(output_path, filename)
    
    try:
        # --- 关键修改：显式指定 Schema ---
        # Polars 的 schema 字典格式： {'col_name': pl.DataType}
        df = pl.DataFrame(data, schema=schema, orient="row")
        
        # 再次确保类型安全（尤其是 behavior_type）
        if "behavior_type" in df.columns:
            df = df.with_columns(pl.col("behavior_type").cast(pl.Int8))
            
        # 使用 snappy 压缩保存
        df.write_parquet(save_path, compression='snappy')
        
    except Exception as e:
        print(f"❌ 保存分块 {filename} 失败: {e}")
        # 打印一行数据样本帮助调试
        if data:
            print(f"   数据样本: {data[0]}")

def process_and_save_parquet(input_files, prefix, schema_dict, row_processor_func, chunk_size=5_000_000):
    """
    读取文件，每积累 chunk_size 行就保存为一个 parquet 文件。
    """
    print(f"\n🚀 开始处理 {prefix} (转换为 Parquet)...")
    
    buffer = []
    chunk_idx = 0
    
    for input_file in input_files:
        print(f"  -> 读取源文件: {os.path.basename(input_file)}")
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in tqdm(f, mininterval=2.0, desc=f"Processing {os.path.basename(input_file)}"):
                line = line.strip()
                if not line: continue
                
                row = row_processor_func(line)
                if row:
                    buffer.append(row)
                
                if len(buffer) >= chunk_size:
                    _save_chunk(buffer, schema_dict, prefix, chunk_idx)
                    buffer = [] 
                    chunk_idx += 1
                    gc.collect()
    
    if buffer:
        _save_chunk(buffer, schema_dict, prefix, chunk_idx)
    
    print(f"✅ {prefix} 全部处理完成！")

# === 3. 行处理逻辑 ===
# 注意：这里我们为了安全，将 ID 类都作为 String 处理，或者确保转换逻辑一致
# 原始数据中 ID 很大，Int64 够用，但为了避免任何科学计数法等问题，String 是最安全的，
# 不过考虑到后续计算效率，用 Int64 也可以，只要保证没有非数字字符。

def process_item_row(line):
    parts = line.split()
    if len(parts) == 3:
        # [id, geo, cat]
        return [int(parts[0]), parts[1], int(parts[2])]
    elif len(parts) == 2:
        # [id, cat] -> geo is None
        return [int(parts[0]), None, int(parts[1])]
    return None

def process_user_row(line):
    parts = line.split()
    # parts: [uid, iid, beh, (geo), cat, date, hour]
    try:
        if len(parts) == 7:
            # 完整数据
            return [
                int(parts[0]),      # user_id
                int(parts[1]),      # item_id
                int(parts[2]),      # behavior_type
                parts[3],           # item_geohash (String or None)
                int(parts[4]),      # item_category
                f"{parts[5]} {parts[6]}" # time (String)
            ]
        elif len(parts) == 6:
            # 缺失 geohash
            return [
                int(parts[0]),
                int(parts[1]),
                int(parts[2]),
                None,               # item_geohash is None
                int(parts[3]),
                f"{parts[4]} {parts[5]}"
            ]
    except ValueError:
        # 防止遇到非数字 ID 报错
        return None
    return None

# === 4. 定义严格的 Schema 并执行 ===

# --- A. Item 表 Schema ---
item_schema = {
    "item_id": pl.Int64,
    "item_geohash": pl.Utf8, # 强制为 String，避免混合类型报错
    "item_category": pl.Int64
}

item_files = glob.glob(os.path.join(dataset_path, "**", "tianchi_fresh_comp_train_item_online.txt"), recursive=True)
if item_files:
    process_and_save_parquet(
        input_files=item_files,
        prefix="train_item",
        schema_dict=item_schema,
        row_processor_func=process_item_row,
        chunk_size=5_000_000
    )

# --- B. User 表 Schema ---
user_schema = {
    "user_id": pl.Int64,
    "item_id": pl.Int64,
    "behavior_type": pl.Int8,
    "item_geohash": pl.Utf8, # 强制为 String
    "item_category": pl.Int64,
    "time": pl.Utf8
}

user_files = []
user_files.extend(glob.glob(os.path.join(dataset_path, "**", "tianchi_fresh_comp_train_user_online_partA.txt"), recursive=True))
user_files.extend(glob.glob(os.path.join(dataset_path, "**", "tianchi_fresh_comp_train_user_online_partB.txt"), recursive=True))

if user_files:
    process_and_save_parquet(
        input_files=user_files,
        prefix="train_user",
        schema_dict=user_schema,
        row_processor_func=process_user_row,
        chunk_size=5_000_000 # 500万行一个分块
    )

In [ ]:
#全局商品（6699万）和目标商品（605万）
import polars as pl
import glob

# 获取所有的 user 和 item 的 parquet 分块文件路径
user_files = glob.glob("/kaggle/input/datasets/lan720/mobileresset/train_user_*.parquet")
item_files = glob.glob("/kaggle/input/datasets/lan720/mobileresset/train_item_*.parquet")

print("🔍 开始统计原始数据的 ID 数量...\n")

# 1. 统计原始唯一的 User ID 数量 (从行为日志表中统计)
if user_files:
    # 使用 scan_parquet 惰性读取，只加载 user_id 这一列，极大节省内存
    unique_users = pl.scan_parquet(user_files).select("user_id").unique().collect().height
    print(f"👤 原始唯一的 user_id 数量: {unique_users}")
else:
    print("❌ 未找到 user_files")

# 2. 统计原始唯一的 Item ID 数量 (即官方给定的商品候选池)
if item_files:
    unique_items = pl.scan_parquet(item_files).select("item_id").unique().collect().height
    print(f"📦 官方商品池中唯一的 item_id 数量: {unique_items}")
else:
    print("❌ 未找到 item_files")

# 3. 用户实际产生过交互的唯一 Item 数量
if user_files:
    interacted_items = pl.scan_parquet(user_files).select("item_id").unique().collect().height
    print(f"🛒 用户日志中实际被交互过的 item_id 数量: {interacted_items}")

In [ ]:
# 🚀 开始极速统计 Parquet 总行数...
# 👤 train_user_*.parquet 文件的总行数: 1,165,522,826
# 📦 train_item_*.parquet 文件的总行数: 6,781,009

import polars as pl
import glob

# 注意：请根据您实际加载的路径选择（可能是 /kaggle/working/ 或 /kaggle/input/...）
user_files = glob.glob("/kaggle/input/datasets/lan720/mobileresset/train_user_*.parquet")
item_files = glob.glob("/kaggle/input/datasets/lan720/mobileresset/train_item_*.parquet")

print("🚀 开始极速统计 Parquet 总行数...\n")

# 1. 统计用户日志表总行数
if user_files:
    # pl.len() 是 Polars 最新的标准统计行数写法，结合 scan_parquet 可以实现“零内存”极速统计
    user_rows = pl.scan_parquet(user_files).select(pl.len()).collect().item()
    print(f"👤 train_user_*.parquet 文件的总行数: {user_rows:,}")
else:
    print("❌ 未找到 train_user_*.parquet 文件")

# 2. 统计商品池表总行数
if item_files:
    item_rows = pl.scan_parquet(item_files).select(pl.len()).collect().item()
    print(f"📦 train_item_*.parquet 文件的总行数: {item_rows:,}")
else:
    print("❌ 未找到 train_item_*.parquet 文件")

In [ ]:
# 1165522826
# 🌍 Geohash 缺失行数: 795826525
# 📊 Geohash 缺失率: 68.28%
# 6781009
# 🌍 Geohash 缺失行数: 5957739
# 📊 Geohash 缺失率: 87.86%

import polars as pl
import glob

user_files = glob.glob("/kaggle/input/datasets/lan720/mobileresset/train_user_*.parquet")

if user_files:
    # 利用 Lazy 模式极速统计缺失值（注意：你在 Cell 2 的 schema 中把它命名为了 item_geohash）
    stats = pl.scan_parquet(user_files).select([
        pl.count().alias("total_rows"),
        pl.col("item_geohash").is_null().sum().alias("missing_rows")
    ]).collect()
    
    total = stats["total_rows"][0]
    missing = stats["missing_rows"][0]
    print(f"{total}")
    print(f"🌍 Geohash 缺失行数: {missing}")
    print(f"📊 Geohash 缺失率: {missing / total * 100:.2f}%")

item_files = glob.glob("/kaggle/input/datasets/lan720/mobileresset/train_item_*.parquet")

if item_files:
    # 利用 Lazy 模式极速统计缺失值（注意：你在 Cell 2 的 schema 中把它命名为了 item_geohash）
    stats = pl.scan_parquet(item_files).select([
        pl.count().alias("total_rows"),
        pl.col("item_geohash").is_null().sum().alias("missing_rows")
    ]).collect()
    
    total = stats["total_rows"][0]
    missing = stats["missing_rows"][0]
    print(f"{total}")
    print(f"🌍 Geohash 缺失行数: {missing}")
    print(f"📊 Geohash 缺失率: {missing / total * 100:.2f}%")

In [1]:
import polars as pl

def agg_u_features(df, windows):
    base_agg = []
    for w in windows:
        wd = f"{w}d"
        in_w = pl.col(f"in_{wd}")
        base_agg.extend([
            # 行为频次：修改为整数列表 [1, 2, 3, 4]
            *[pl.col("behavior_type").filter(in_w & (pl.col("behavior_type") == bt)).len().alias(f"u_behav_{bt}_{wd}") for bt in [1, 2, 3, 4]],
            # 加权分
            (pl.col("base_weight") * in_w).sum().alias(f"u_weighted_{wd}"),
            # 多样性
            pl.col("item_id").filter(in_w).n_unique().alias(f"u_item_nuniq_{wd}"),
            pl.col("item_category").filter(in_w).n_unique().alias(f"u_cat_nuniq_{wd}"),
            # 活跃天数
            pl.col("time").filter(in_w).dt.date().n_unique().alias(f"u_active_days_{wd}"),
            # 购买天数：修改为整数 4
            pl.col("time").filter(in_w & (pl.col("behavior_type") == 4)).dt.date().n_unique().alias(f"u_buy_days_{wd}"),
            # 小时均值
            pl.col("hour").filter(in_w).mean().alias(f"u_hour_mean_{wd}"),
            # 平滑计数（用于后续 ratio）：修改为整数 4
            (pl.col("behavior_type").filter(in_w & (pl.col("behavior_type") == 4)).len() + 1).alias(f"buy_cnt_{wd}"),
            (pl.col("behavior_type").filter(in_w).len() + 1).alias(f"total_cnt_{wd}"),
            (pl.col("item_id").filter(in_w & (pl.col("behavior_type") == 4)).n_unique() + 1).alias(f"buy_item_{wd}"),
            (pl.col("item_id").filter(in_w).n_unique() + 1).alias(f"interact_item_{wd}"),
        ])

    # === 新增：用户视角的全局衰减特征 ===
    decay_features = [
        # 用户近期的整体活跃度衰减分（分值越高说明刚刚越活跃）
        pl.col("decay_weight_inv").sum().alias("u_total_decay_inv_score"),
        pl.col("decay_weight_exp").filter(pl.col("behavior_type") == 1).sum().alias("u_click_decay_exp_score"),
    ]
    
    df_agg = df.group_by("user_id").agg(base_agg + decay_features)

    # 计算 ratio 并 drop中间列
    ratio_exprs = []
    drop_cols = []
    for w in windows:
        wd = f"{w}d"
        ratio_exprs.extend([
            (pl.col(f"buy_cnt_{wd}") / pl.col(f"total_cnt_{wd}")).alias(f"u_buy_ratio_{wd}"),
            (pl.col(f"buy_item_{wd}") / pl.col(f"interact_item_{wd}")).alias(f"u_buy_item_ratio_{wd}"),
        ])
        drop_cols.extend([f"buy_cnt_{wd}", f"total_cnt_{wd}", f"buy_item_{wd}", f"interact_item_{wd}"])

    return df_agg.with_columns(ratio_exprs).drop(drop_cols)

In [2]:
def agg_ui_features(df, prom_date, windows):
    base_agg = []
    for w in windows:
        wd = f"{w}d"
        in_w = pl.col(f"in_{wd}")
        base_agg.extend([
            # 行为频次：修改为整数列表 [1, 2, 3, 4]
            *[pl.col("behavior_type").filter(in_w & (pl.col("behavior_type") == bt)).len().alias(f"ui_behav_{bt}_{wd}") for bt in [1, 2, 3, 4]],
            (pl.col("base_weight") * in_w).sum().alias(f"ui_weighted_{wd}"),
            pl.col("time").filter(in_w).dt.date().n_unique().alias(f"ui_active_days_{wd}"),
            pl.col("hour").filter(in_w).mean().alias(f"ui_hour_mean_{wd}"),
        ])
    # === 新增：全局时间衰减特征 (不限具体窗口) ===
    decay_features = [
        # 总行为衰减得分
        pl.col("decay_weight_inv").sum().alias("ui_total_decay_inv_score"),
        pl.col("decay_weight_exp").sum().alias("ui_total_decay_exp_score"),
        # 核心行为(加购/收藏)的衰减得分，极大概率成为强特征
        pl.col("decay_weight_exp").filter(pl.col("behavior_type").is_in([2, 3])).sum().alias("ui_cart_fav_decay_score")
    ]

    # === 促销特征（仅 in_7d）===
    in_7d = pl.col("in_7d")
    # 修改比较值为整数 2 和 3
    new_features = [
        pl.col("behavior_type")
          .filter(in_7d & (pl.col("time").dt.date() == prom_date) & (pl.col("behavior_type") == 2))
          .len().gt(0).cast(pl.Int8).alias("ui_fav_in_prom"),
        pl.col("behavior_type")
          .filter(in_7d & (pl.col("time").dt.date() == prom_date) & (pl.col("behavior_type") == 3))
          .len().gt(0).cast(pl.Int8).alias("ui_cart_in_prom"),
        pl.col("behavior_type")
          .filter(in_7d & (pl.col("time").dt.date() != prom_date) & (pl.col("behavior_type") == 2))
          .len().gt(0).cast(pl.Int8).alias("ui_fav_in_normal"),
        pl.col("behavior_type")
          .filter(in_7d & (pl.col("time").dt.date() != prom_date) & (pl.col("behavior_type") == 3))
          .len().gt(0).cast(pl.Int8).alias("ui_cart_in_normal"),
    ]

    return df.group_by(["user_id", "item_id"]).agg(base_agg + decay_features + new_features)

In [3]:
def agg_uc_features(df, windows):
    agg_exprs = []
    for w in windows:
        wd = f"{w}d"
        in_w = pl.col(f"in_{wd}")
        agg_exprs.extend([
            pl.col("item_id").filter(in_w).n_unique().alias(f"uc_item_nuniq_{wd}"),
            # 行为频次：修改为整数列表 [1, 2, 3, 4]
            *[pl.col("behavior_type").filter(in_w & (pl.col("behavior_type") == bt)).len().alias(f"uc_behav_{bt}_{wd}") for bt in [1, 2, 3, 4]],
            (pl.col("base_weight") * in_w).sum().alias(f"uc_weighted_{wd}"),
            pl.col("time").filter(in_w).dt.date().n_unique().alias(f"uc_active_days_{wd}"),
            pl.col("hour").filter(in_w).mean().alias(f"uc_hour_mean_{wd}"),
        ])
    # === 新增：类别级衰减 ===
    decay_features = [
        pl.col("decay_weight_inv").sum().alias("uc_total_decay_inv_score"),
        # 用户最近对该类别的加购/收藏意图强度
        pl.col("decay_weight_exp").filter(pl.col("behavior_type").is_in([2, 3])).sum().alias("uc_cart_fav_decay_score")
    ]
    return df.group_by(["user_id", "item_category"]).agg(agg_exprs + decay_features)

In [4]:
import polars as pl
from datetime import datetime, date, timedelta
from typing import List, Set
import os
import gc

# 确保全局变量 WINDOWS 已定义
WINDOWS = [1, 2, 4, 7, 14, 21, 28]

def get_item_category_feat(file_paths: List[str], label_date: str, item_set: Set[int], skip_prom: bool):
    prom_str = "2014-12-12"
    label_dt = datetime.strptime(label_date, "%Y-%m-%d").date()
    min_date = label_dt - timedelta(days=29)
    min_date_str = min_date.strftime("%Y-%m-%d")
    
    valid_paths = [fp for fp in file_paths if os.path.exists(fp)]
    if not valid_paths:
        return pl.DataFrame(schema={"item_id": pl.Int64, "item_category": pl.Int64})

    print(f"[ItemFeat] 1. Scanning {len(valid_paths)} files using String Filter...")
    lf = pl.scan_parquet(valid_paths)
    lf = lf.filter((pl.col("time") >= min_date_str) & (pl.col("time") < label_date))
    
    if skip_prom:
        lf = lf.filter(~pl.col("time").str.starts_with(prom_str))

    target_items = list(item_set)
    lf = lf.filter(pl.col("item_id").is_in(target_items))
    
    print("[ItemFeat] 2. Computing Base Subset in Memory...")
    # 🚨 核心修复 1：提取所需的最小列，并直接拉进内存物化，切断 Lazy 图
    df_base = lf.select(["user_id", "item_id", "item_category", "behavior_type", "time"]).collect(engine="streaming")
    
    # 在内存 Eager 模式下进行日期转换和窗口标记
    df_base = df_base.with_columns(pl.col("time").str.slice(0, 10).cast(pl.Date).alias("date"))
    
    def get_start_date(target_days: int) -> date:
        start = label_dt - timedelta(days=target_days)
        if not skip_prom:
            return start
        end = label_dt - timedelta(days=1)
        if start <= date(2014, 12, 12) <= end:
            return start - timedelta(days=1)
        return start

    start_dates_map = {f"{w}d": get_start_date(w) for w in WINDOWS}
    window_exprs = [(pl.col("date") >= start_dates_map[f"{w}d"]).alias(f"in_{w}d") for w in WINDOWS]
    df_base = df_base.with_columns(window_exprs)

    print(f"[ItemFeat] 3. Aggregating Features (Base rows: {df_base.height})...")
    
    # 预备聚合表达式
    i_agg_exprs = []
    c_agg_exprs = []
    for w in WINDOWS:
        wd = f"{w}d"
        in_w = pl.col(f"in_{wd}")
        i_agg_exprs.extend([
            *[pl.col("behavior_type").filter(in_w & (pl.col("behavior_type") == bt)).len().alias(f"i_behav_{bt}_{wd}") for bt in [1, 2, 3, 4]],
            pl.col("user_id").filter(in_w).n_unique().alias(f"i_interact_users_{wd}"),
            pl.col("user_id").filter(in_w & (pl.col("behavior_type") == 4)).n_unique().alias(f"i_buy_users_{wd}")
        ])
        c_agg_exprs.extend([
            pl.col("item_id").filter(in_w).n_unique().alias(f"c_interacted_items_{wd}"),
            *[pl.col("behavior_type").filter(in_w & (pl.col("behavior_type") == bt)).len().alias(f"c_behav_{bt}_{wd}") for bt in [1, 2, 3, 4]],
            pl.col("user_id").filter(in_w).n_unique().alias(f"c_interact_users_{wd}"),
            pl.col("user_id").filter(in_w & (pl.col("behavior_type") == 4)).n_unique().alias(f"c_buy_users_{wd}")
        ])

    # 🚨 核心修复 2：在内存中分别聚合，速度极快且不堆积内存垃圾
    i_features = df_base.group_by(["item_id", "item_category"]).agg(i_agg_exprs)
    c_features = df_base.group_by("item_category").agg(c_agg_exprs)

    # 释放基础日志表内存
    del df_base
    gc.collect()

    print("[ItemFeat] 4. Merging and Ranking...")
    df_ic = i_features.join(c_features, on="item_category", how="left")
    
    # 释放中间表
    del i_features, c_features
    gc.collect()

    # 计算比率
    ratio_exprs = []
    for w in WINDOWS:
        wd = f"{w}d"
        ratio_exprs.extend([
            ((pl.col(f"i_behav_4_{wd}") + 1) / (pl.col(f"c_behav_4_{wd}") + 1)).alias(f"ic_buy_ratio_{wd}"),
            ((pl.col(f"i_buy_users_{wd}") + 1) / (pl.col(f"i_interact_users_{wd}") + 1)).alias(f"i_buy_user_ratio_{wd}"),
            ((pl.col(f"c_buy_users_{wd}") + 1) / (pl.col(f"c_interact_users_{wd}") + 1)).alias(f"c_buy_user_ratio_{wd}")
        ])
    df_ic = df_ic.with_columns(ratio_exprs)

    # 🚨 核心修复 3：分步执行排名计算。如果一次性丢进所有的 over，多线程上下文切换会导致内存毛刺
    for w in WINDOWS:
        wd = f"{w}d"
        buy_col = f"i_behav_4_{wd}"
        df_ic = df_ic.with_columns(
            ((pl.len().over("item_category") - 
              pl.col(buy_col).rank(method="dense", descending=True).over("item_category") + 1) 
             / pl.len().over("item_category"))
            .alias(f"ic_buy_norm_inv_rank_{wd}")
        )

    return df_ic.fill_null(0)

In [5]:
import polars as pl
from datetime import datetime, timedelta

def build_global_top_items_per_category(user_files, target_date_str, item_set, top_k=3):
    print("⏳ 预计算 U2C2I 全局类别热销表...")
    
    # === 优化 1：限制时间窗口，仅统计最近 7 天的热度，直接砍掉 70% 的无用计算 ===
    target_dt = datetime.strptime(target_date_str, "%Y-%m-%d").date()
    min_dt_str = (target_dt - timedelta(days=7)).strftime("%Y-%m-%d")
    
    lf = pl.scan_parquet(user_files)
    
    # 增加下界过滤
    lf = lf.filter((pl.col("time") >= min_dt_str) & (pl.col("time") < target_date_str))
    
    # 仅限官方商品池中的商品
    target_items = list(item_set)
    lf = lf.filter(pl.col("item_id").is_in(target_items))
    
    # 统计行为频次作为热度
    cat_item_hot = lf.group_by(["item_category", "item_id"]).agg(
        pl.len().alias("hot_score")
    )
    
    # === 优化 2：弃用 sort().head()，使用内部极速的最小堆 top_k_by ===
    top_items_df = (
        cat_item_hot
        .group_by("item_category")
        .agg(
            pl.col("item_id").top_k_by("hot_score", k=top_k)
        )
        .explode("item_id") # 将 list 展开为单行
        .collect() # 触发计算
    )
    
    print(f"✅ 类别热销表构建完毕，覆盖类别数: {top_items_df['item_category'].n_unique()}")
    return top_items_df

In [6]:
import polars as pl
import numpy as np
import scipy.sparse as sp
from datetime import datetime, timedelta
import gc

def build_sparse_itemcf_sim_table(user_files, target_date_str, item_set, days_lookback=4, top_k=20):
    print(f"⏳ 开始预计算 ItemCF 相似度 (极致防爆版, Lookback: {days_lookback} days)...")
    
    target_dt = datetime.strptime(target_date_str, "%Y-%m-%d").date()
    min_dt_str = (target_dt - timedelta(days=days_lookback)).strftime("%Y-%m-%d")
    
    df_base = pl.scan_parquet(user_files).filter(
        (pl.col("time") >= min_dt_str) & (pl.col("time") < target_date_str) & 
        (pl.col("behavior_type").is_in([2,3,4])) 
    ).select(["user_id", "item_id"]).unique().collect()
    
    # === 优化 1：极致截断，剔除度数为1的无效节点（不影响最终效果，但能狂降内存） ===
    user_counts = df_base.group_by("user_id").agg(pl.len().alias("u_cnt"))
    item_counts = df_base.group_by("item_id").agg(pl.len().alias("i_cnt"))
    
    # u_cnt >= 2：没共同交互的用户无协同意义；<= 50：过滤重度活跃机器人
    # i_cnt >= 2：只有一个人的商品无共现意义；<= 200：过滤全民无脑爆款
    df_base = df_base.join(user_counts.filter((pl.col("u_cnt") >= 2) & (pl.col("u_cnt") <= 50)), on="user_id") \
                     .join(item_counts.filter((pl.col("i_cnt") >= 2) & (pl.col("i_cnt") <= 200)), on="item_id") \
                     .select(["user_id", "item_id"])
                     
    del user_counts, item_counts
    gc.collect()

    # ID 转连续索引
    unique_users = df_base.select("user_id").unique().with_row_index("u_idx")
    unique_items = df_base.select("item_id").unique().with_row_index("i_idx")

    df_mapped = df_base.join(unique_users, on="user_id").join(unique_items, on="item_id")

    rows = df_mapped["u_idx"].to_numpy()
    cols = df_mapped["i_idx"].to_numpy()
    data = np.ones(len(rows), dtype=np.float32)

    num_users = unique_users.height
    num_items = unique_items.height
    
    del df_mapped, df_base
    gc.collect()

    print(f"   -> 截断后图矩阵: {num_users} Users x {num_items} Items")
    
    user_item_matrix = sp.csr_matrix((data, (rows, cols)), shape=(num_users, num_items))
    del rows, cols, data
    gc.collect()

    # === 优化 2：内存替换优化，提前缩放列权重，避免后续庞大分配 ===
    item_degrees = np.array(user_item_matrix.sum(axis=0)).flatten()
    inv_degree = 1.0 / np.sqrt(item_degrees + 1e-9)
    
    # 使用 multiply 广播，零分配开销
    R_scaled = user_item_matrix.multiply(inv_degree).tocsr()
    del user_item_matrix, item_degrees, inv_degree
    gc.collect()

    # === 优化 3：分块点乘，死死卡住内存瓶颈 ===
    print("   -> 矩阵分块点乘中 (内存锁定开启)...")
    R_scaled_T = R_scaled.T.tocsr()
    
    chunk_size = 50000  # 每次仅算 5 万个商品的相似度
    rows_list, cols_list, data_list = [], [], []
    
    # 进度提示辅助
    total_chunks = (num_items // chunk_size) + 1
    
    for idx, start_idx in enumerate(range(0, num_items, chunk_size)):
        if idx % 5 == 0:
            print(f"      - 处理进度: {idx}/{total_chunks} blocks")
            
        end_idx = min(start_idx + chunk_size, num_items)
        
        # 只取出 50000 行的子集作点乘，结果矩阵 S_chunk 极小！
        S_chunk = R_scaled_T[start_idx:end_idx].dot(R_scaled)
        
        # 清空自相似度（对角线）
        S_chunk.setdiag(0)
        S_chunk.eliminate_zeros()
        
        # 取出每一行的 Top-K
        for i in range(S_chunk.shape[0]):
            r_start = S_chunk.indptr[i]
            r_end = S_chunk.indptr[i+1]
            if r_end == r_start:
                continue
                
            row_data = S_chunk.data[r_start:r_end]
            row_cols = S_chunk.indices[r_start:r_end]
            
            # 使用 np.argpartition 极速提取 Top K
            if len(row_data) > top_k:
                top_idx = np.argpartition(row_data, -top_k)[-top_k:]
                rows_list.extend([start_idx + i] * top_k)
                cols_list.extend(row_cols[top_idx])
                data_list.extend(row_data[top_idx])
            else:
                rows_list.extend([start_idx + i] * len(row_data))
                cols_list.extend(row_cols)
                data_list.extend(row_data)
                
        # 强制销毁中间块
        del S_chunk
        gc.collect()

    del R_scaled, R_scaled_T
    gc.collect()

    # 结果装入 Polars 
    sim_df = pl.DataFrame({
        "i_idx": np.array(rows_list, dtype=np.uint32),
        "sim_idx": np.array(cols_list, dtype=np.uint32),
        "sim_score": np.array(data_list, dtype=np.float32)
    })
    
    del rows_list, cols_list, data_list
    gc.collect()

    # 映射回原始 item_id 并合并候选集
    print("   -> 匹配全局商品字典...")
    i_idx_map = unique_items.rename({"i_idx": "i_idx", "item_id": "item"})
    sim_idx_map = unique_items.rename({"i_idx": "sim_idx", "item_id": "sim_item"})

    target_items = list(item_set)
    
    topk_sim = sim_df.lazy() \
        .join(i_idx_map.lazy(), on="i_idx") \
        .join(sim_idx_map.lazy(), on="sim_idx") \
        .select(["item", "sim_item", "sim_score"]) \
        .filter(pl.col("sim_item").is_in(target_items)) \
        .collect()

    print(f"✅ ItemCF 计算彻底完成，有效关联商品数: {topk_sim['item'].n_unique()}")
    return topk_sim

In [7]:
def build_global_hot_items(user_files, target_date_str, item_set, top_k=50):
    print("⏳ 预计算全局热卖兜底商品池...")
    target_dt = datetime.strptime(target_date_str, "%Y-%m-%d").date()
    min_dt_str = (target_dt - timedelta(days=3)).strftime("%Y-%m-%d")
    
    lf = pl.scan_parquet(user_files).filter(
        (pl.col("time") >= min_dt_str) & 
        (pl.col("time") < target_date_str) &
        (pl.col("behavior_type") == 4) &  # 仅统计真实购买
        (pl.col("item_id").is_in(list(item_set)))
    )
    
    top_items = (
        lf.group_by("item_id")
        .agg(pl.len().alias("buy_cnt"))
        .sort("buy_cnt", descending=True)
        .head(top_k)
        .select("item_id")
        .collect()
    )
    return top_items

In [8]:
import polars as pl
from datetime import datetime, date, timedelta
from typing import List, Set
import os
import gc
import traceback

# 确保全局变量 WINDOWS 已定义
WINDOWS = [1, 2, 4, 7, 14, 21, 28]

def get_sample_u_feat(file_paths: List[str], label_date: str, item_set: Set[int], 
                       itemcf_sim_df: pl.DataFrame, 
                       global_hot_items_df: pl.DataFrame, 
                       is_label: bool, skip_prom: bool):
    weight_map = {1: 1, 2: 2, 3: 3, 4: 10}
    prom = date(2014, 12, 12)
    label_date_obj = datetime.strptime(label_date, "%Y-%m-%d").date()

    def get_start_date(target_days: int) -> date:
        if not skip_prom:
            return label_date_obj - timedelta(days=target_days)
        start = label_date_obj - timedelta(days=target_days)
        end = label_date_obj - timedelta(days=1)
        if start <= prom <= end:
            return start - timedelta(days=1)
        return start

    start_dates = {f"{w}d": get_start_date(w) for w in WINDOWS}
    
    BATCH_SIZE = 2  
    tmp_dir = f"./tmp_ufeat_{label_date}"
    os.makedirs(tmp_dir, exist_ok=True)
    
    valid_paths = [fp for fp in file_paths if os.path.exists(fp)]
    total_files = len(valid_paths)
    
    print(f"开始处理 {total_files} 个文件，Batch Size = {BATCH_SIZE}...")

    for i in range(0, total_files, BATCH_SIZE):
        batch_paths = valid_paths[i : i + BATCH_SIZE]
        try:
            lf = pl.scan_parquet(batch_paths)
            lf = lf.with_columns((pl.col("time") + ":00").str.to_datetime("%Y-%m-%d %H:%M"))

            cand_start = start_dates["7d"] 
            
            # --- 路 1: 历史行为召回 (U2I) - 占比级：全量保留 ---
            candidate_filter = (
                (pl.col("time").dt.date() >= cand_start) &
                (pl.col("time").dt.date() < label_date_obj) &
                (pl.col("behavior_type").is_in([1, 2, 3])) & 
                (pl.col("item_id").is_in(list(item_set))) 
            )
            if skip_prom:
                candidate_filter = candidate_filter & (pl.col("time").dt.date() != prom)

            df_u2i = lf.filter(candidate_filter) \
                       .select(["user_id", "item_id", "item_category"]) \
                       .unique() \
                       .with_columns(pl.lit(1).cast(pl.Int8).alias("recall_u2i")) \
                       .collect()

            # --- 路 2: itemcf 协同过滤召回 (U2I2I) - 占比级：Top 30 ---
            trigger_filter = (
                (pl.col("time").dt.date() >= cand_start) &
                (pl.col("time").dt.date() < label_date_obj) &
                (pl.col("behavior_type").is_in([2, 3, 4]))
            )
            if skip_prom:
                trigger_filter = trigger_filter & (pl.col("time").dt.date() != prom)

            user_trigger_items = (
                lf.filter(trigger_filter)
                .group_by(["user_id", "item_id"])
                .agg(pl.len().alias("trigger_weight"))
                .filter(pl.col("trigger_weight").rank(descending=True).over("user_id") <= 10) 
                .select(["user_id", "item_id", "trigger_weight"])
            )

            # 【比例优化】按 sim_score * trigger_weight 打分，截断至最多 30 个
            df_itemcf = user_trigger_items.join(
                itemcf_sim_df.lazy(), left_on="item_id", right_on="item", how="inner"
            ).with_columns(
                (pl.col("trigger_weight") * pl.col("sim_score")).alias("final_score")
            ).group_by(["user_id", "sim_item"]).agg(
                pl.col("final_score").sum().alias("total_score")
            ).filter(
                pl.col("total_score").rank(descending=True).over("user_id") <= 30
            ).select([
                pl.col("user_id"),
                pl.col("sim_item").alias("item_id")
            ]).with_columns(pl.lit(1).cast(pl.Int8).alias("recall_itemcf"))
            
            item_cat_map = lf.select(["item_id", "item_category"]).unique().lazy()
            df_itemcf = df_itemcf.join(item_cat_map, on="item_id", how="inner").collect(engine="streaming")

            # --- 路 3: 全局热卖兜底召回 (Global Hot) - 极度克制版 ---
            # 提取本批次所有活跃用户
            active_users = lf.select("user_id").unique().collect()
            
            # 统计当前已被 U2I 和 ItemCF 覆盖的候选集数量
            current_cands = pl.concat([
                df_u2i.select("user_id"), 
                df_itemcf.select("user_id")
            ]).group_by("user_id").agg(pl.len().alias("cnt"))
            
            # 识别出候选商品数 < 10 的冷启动/低活用户
            rich_users = current_cands.filter(pl.col("cnt") >= 10).select("user_id")
            need_padding_users = active_users.join(rich_users, on="user_id", how="anti")
            
            # 仅对这部分缺乏候选的用户推送 Top 5 全局热卖
            if need_padding_users.height > 0:
                df_global = need_padding_users.lazy().join(global_hot_items_df.lazy(), how="cross") \
                                      .with_columns(pl.lit(1).cast(pl.Int8).alias("recall_global"))
                
                item_cat_map = lf.select(["item_id", "item_category"]).unique()
                df_global = df_global.join(item_cat_map.lazy(), on="item_id", how="inner").collect(engine="streaming")
            else:
                df_global = pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "item_category": pl.Int64, "recall_global": pl.Int8})

            # --- 多路融合与去重 ---
            if df_u2i.is_empty() and df_itemcf.is_empty() and df_global.is_empty():
                continue
                
            df_candidate = pl.concat([df_u2i, df_itemcf, df_global], how="diagonal") \
                             .group_by(["user_id", "item_id", "item_category"]) \
                             .agg([
                                 pl.col("recall_u2i").max().fill_null(0).cast(pl.Int8),
                                 pl.col("recall_itemcf").max().fill_null(0).cast(pl.Int8),
                                 pl.col("recall_global").max().fill_null(0).cast(pl.Int8)
                             ])
                             
            if df_candidate.is_empty(): continue

            # === 打标签逻辑 ===
            if is_label:
                df_buy = lf.filter(
                    (pl.col("time").dt.date() == label_date_obj) &
                    (pl.col("behavior_type") == 4)
                ).select(["user_id", "item_id"]).unique().with_columns(pl.lit(1).alias("is_buy")).collect()

                df_labeled = df_candidate.join(df_buy, on=["user_id", "item_id"], how="left") \
                                         .with_columns(pl.col("is_buy").fill_null(0).cast(pl.Int8).alias("label")) \
                                         .drop("is_buy")
            else:
                df_labeled = df_candidate

            # === Step 2: 特征计算 ===
            feat_filter = (
                (pl.col("time").dt.date() >= start_dates["28d"]) &
                (pl.col("time").dt.date() < label_date_obj)
            )
            if skip_prom:
                feat_filter = feat_filter & (pl.col("time").dt.date() != prom)

            cand_user_ids = df_labeled.select("user_id").unique()
            lf_feat = lf.filter(feat_filter).join(cand_user_ids.lazy(), on="user_id", how="semi")

            window_exprs = []
            for w in WINDOWS:
                col_name = f"in_{w}d"
                window_exprs.append((pl.col("time").dt.date() >= start_dates[f"{w}d"]).alias(col_name))
            
            target_dt = datetime.combine(label_date_obj, datetime.min.time())

            lf_log = lf_feat.with_columns([
                pl.col("time").dt.hour().alias("hour"),
                pl.col("behavior_type").replace_strict(weight_map, default=1).cast(pl.Float32).alias("base_weight"),
                (pl.lit(target_dt) - pl.col("time")).dt.total_hours().cast(pl.Float32).alias("delta_hours"),
                *window_exprs
            ]).with_columns([
                (pl.col("base_weight") / (1.0 + pl.col("delta_hours") / 24.0)).alias("decay_weight_inv"),
                (pl.col("base_weight") * (-pl.col("delta_hours") / 48.0).exp()).alias("decay_weight_exp")
            ])
            
            df_log_filtered = lf_log.collect(engine="streaming")
            if df_log_filtered.is_empty():
                continue

            prom_date = date(2014, 12, 12)

            u_feats = agg_u_features(df_log_filtered, WINDOWS)
            ui_feats = agg_ui_features(df_log_filtered, prom_date, WINDOWS)
            uc_feats = agg_uc_features(df_log_filtered, WINDOWS)

            df_out = (
                df_labeled.lazy()
                .join(u_feats.lazy(), on="user_id", how="left")
                .join(ui_feats.lazy(), on=["user_id", "item_id"], how="left")
                .join(uc_feats.lazy(), on=["user_id", "item_category"], how="left")
            ).collect()

            ratio_exprs = []
            for w in WINDOWS:
                window = f"{w}d"
                ui_col = f"ui_weighted_{window}"
                uc_col = f"uc_weighted_{window}"
                ratio_exprs.append(
                    ((pl.col(ui_col).fill_null(0) + 1) / (pl.col(uc_col).fill_null(0) + 1))
                    .alias(f"ui_uc_weighted_ratio_{window}")
                )
            df_out = df_out.with_columns(ratio_exprs)

            weight_cols = [f"ui_weighted_{w}d" for w in WINDOWS]
            ui_weight_df = ui_feats.select(["user_id", "item_id"] + weight_cols)

            rank_df = df_labeled.select(["user_id", "item_category", "item_id"]) \
                .join(ui_weight_df, on=["user_id", "item_id"], how="left")

            rank_exprs = []
            for w in WINDOWS:
                window = f"{w}d"
                wcol = f"ui_weighted_{window}"
                rank_exprs.append(
                    ((pl.len().over(["user_id", "item_category"]) - 
                      pl.col(wcol).fill_null(0).rank(method="dense", descending=True).over(["user_id", "item_category"]) + 1) 
                     / pl.len().over(["user_id", "item_category"]))
                    .alias(f"ui_norm_inv_rank_in_uc_{window}")
                )
            
            rank_df = rank_df.with_columns(rank_exprs)

            rank_select_cols = ["user_id", "item_category", "item_id"] + [f"ui_norm_inv_rank_in_uc_{w}d" for w in WINDOWS]
            
            df_out = df_out.join(
                rank_df.select(rank_select_cols),
                on=["user_id", "item_category", "item_id"],
                how="left"
            )
            df_out = df_out.fill_null(0)
            
            tmp_file = os.path.join(tmp_dir, f"batch_{i}.parquet")
            df_out.write_parquet(tmp_file)
            
            del df_log_filtered, lf, lf_log, df_out
            gc.collect()

        except Exception as e:
            print(f"Error processing batch starting at {i}: {e}")
            traceback.print_exc()
            continue

    print("✅ 所有用户侧特征批次落盘完成！")
    return pl.scan_parquet(os.path.join(tmp_dir, "*.parquet"))

In [9]:
# 假设 user_files 和 item_set 已在内存中
import glob
import polars as pl
import pyarrow.parquet as pq
import shutil

user_files = sorted(glob.glob("Dataset/train_user_*.parquet"))
item_files = sorted(glob.glob("Dataset/train_item_*.parquet"))
item_set = set()
for f in item_files:
    item_set.update(pl.read_parquet(f, columns=["item_id"])["item_id"].unique().to_list())

def generate_dataset(target_date, is_label_data, save_name):
    print(f"\n🚀 正在生成 {save_name} (Label Date: {target_date})...")
    
    print("  -> 0.1 Pre-computing Global Hot Items...")
    global_hot_items = build_global_hot_items(
        user_files=user_files, target_date_str=target_date, item_set=item_set, top_k=5 # 【修改点】仅取 Top 5 进行兜底
    )
    
    print("  -> 0.2 Pre-computing ItemCF Similarities...")
    itemcf_sim_df = build_sparse_itemcf_sim_table(
        user_files=user_files, target_date_str=target_date, item_set=item_set, 
        days_lookback=7, top_k=100
    )
    
    print("  -> 1. User Features (Multi-Path Recall)...")
    u_feat_lazy = get_sample_u_feat(
        file_paths=user_files, label_date=target_date, item_set=item_set, 
        itemcf_sim_df=itemcf_sim_df, 
        global_hot_items_df=global_hot_items, 
        is_label=is_label_data, skip_prom=True
    )
    
    print("  -> 2. Item Features...")
    current_item_set = set(
        u_feat_lazy.select("item_id").unique().collect(engine="streaming")["item_id"].to_list()
    )
    ic_feat = get_item_category_feat(
        file_paths=user_files, label_date=target_date, item_set=current_item_set, skip_prom=True
    )

    print("  -> 3. Chunked Merging & Saving...")
    u_feat_cols = u_feat_lazy.collect_schema().names()
    cols_to_drop = [c for c in ic_feat.columns if c in u_feat_cols and c not in ['item_id', 'item_category']]
    if cols_to_drop: 
        ic_feat = ic_feat.drop(cols_to_drop)
        
    output_path = f"./{save_name}.parquet"
    chunk_size = 2_000_000
    
    total_rows = u_feat_lazy.select(pl.len()).collect().item()
    print(f"   -> 准备融合总行数: {total_rows}")
    
    writer = None
    for offset in range(0, total_rows, chunk_size):
        df_chunk = (
            u_feat_lazy.slice(offset, chunk_size)
            .collect(engine="streaming")
            .join(ic_feat, on=['item_id', 'item_category'], how='left')
            .to_arrow()
        )
        
        if writer is None:
            writer = pq.ParquetWriter(output_path, df_chunk.schema)
        writer.write_table(df_chunk)
        
    if writer:
        writer.close()
        
    print(f"✅ 已分块保存完毕: {output_path} (Total Rows: {total_rows})")
    
    del u_feat_lazy, ic_feat
    gc.collect()
    shutil.rmtree(f"./tmp_ufeat_{target_date}", ignore_errors=True)

generate_dataset("2014-12-17", is_label_data=True, save_name="train_set_1217_2026-0331")
generate_dataset("2014-12-18", is_label_data=True, save_name="val_set_1218_2026-0331")
generate_dataset("2014-12-19", is_label_data=False, save_name="test_set_1219_2026-0331")


🚀 正在生成 train_set_1217_2026-0331 (Label Date: 2014-12-17)...
  -> 0.1 Pre-computing Global Hot Items...
⏳ 预计算全局热卖兜底商品池...
  -> 0.2 Pre-computing ItemCF Similarities...
⏳ 开始预计算 ItemCF 相似度 (极致防爆版, Lookback: 7 days)...
   -> 截断后图矩阵: 653254 Users x 1499163 Items
   -> 矩阵分块点乘中 (内存锁定开启)...
      - 处理进度: 0/30 blocks
      - 处理进度: 5/30 blocks
      - 处理进度: 10/30 blocks
      - 处理进度: 15/30 blocks
      - 处理进度: 20/30 blocks
      - 处理进度: 25/30 blocks
   -> 匹配全局商品字典...
✅ ItemCF 计算彻底完成，有效关联商品数: 908215
  -> 1. User Features (Multi-Path Recall)...
开始处理 234 个文件，Batch Size = 2...
✅ 所有用户侧特征批次落盘完成！
  -> 2. Item Features...
[ItemFeat] 1. Scanning 234 files using String Filter...
[ItemFeat] 2. Computing Base Subset in Memory...
[ItemFeat] 3. Aggregating Features (Base rows: 81739466)...
[ItemFeat] 4. Merging and Ranking...
  -> 3. Chunked Merging & Saving...
   -> 准备融合总行数: 18094790
✅ 已分块保存完毕: ./train_set_1217_2026-0331.parquet (Total Rows: 18094790)

🚀 正在生成 val_set_1218_2026-0331 (Label Date: 2014-12-18).

In [10]:
import pyarrow.parquet as pq
import lightgbm as lgb
import pandas as pd
import numpy as np
import gc
import os

# ==========================================
# 1. 全局配置
# ==========================================
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "num_leaves": 31,           # 严格锁死在 31，严禁放大
    "learning_rate": 0.05,
    "min_data_in_leaf": 50,     # 回调至 2-21 的最佳值
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 1e-2,
    "lambda_l2": 1e-1,
    "verbose": -1,
    "seed": 42,
    "n_jobs": -1,
}

SUBMIT_K = 35000

TRAIN_PATH = "./train_set_1217_2026-0331.parquet"
VAL_PATH = "./val_set_1218_2026-0331.parquet"
TEST_PATH = "./test_set_1219_2026-0331.parquet"
OUTPUT_DIR = "."

# 【核心修正】彻底封杀 item_category，完全依赖 uc_ 交叉特征
IGNORE_COLS = ["user_id", "item_id", "label", "time", "item_category"]
DROP_FILE_NAME = "features_to_drop-v3.csv" # 启用全新黑名单，避免被上轮错误诊断污染

# ==========================================
# 2. 核心函数：动态下采样读取
# ==========================================
def load_and_downsample(file_path, neg_sample_rate):
    print(f"  -> 读取文件: {os.path.basename(file_path)} | 采样率: {neg_sample_rate}")
    parquet_file = pq.ParquetFile(file_path)
    chunks = []

    for batch in parquet_file.iter_batches(batch_size=500000):
        df_chunk = batch.to_pandas()

        for col in df_chunk.columns:
            if df_chunk[col].dtype == "float64":
                df_chunk[col] = df_chunk[col].astype("float32")
            elif df_chunk[col].dtype == "int64":
                df_chunk[col] = df_chunk[col].astype("int32")

        pos_df = df_chunk[df_chunk["label"] == 1]
        neg_df = df_chunk[df_chunk["label"] == 0]

        neg_sampled = neg_df.sample(frac=neg_sample_rate, random_state=42)
        chunks.append(pd.concat([pos_df, neg_sampled], ignore_index=True))

        del df_chunk, pos_df, neg_df, neg_sampled, batch
        gc.collect()

    df_final = pd.concat(chunks, ignore_index=True)
    print(f"     ✅ 加载完成. 最终行数: {len(df_final)}")
    return df_final


# ==========================================
# 3. 实验主流程
# ==========================================
def run_ablation_experiment(neg_rate, ignore_cols=None):
    local_ignore_cols = list(ignore_cols) if ignore_cols else []

    print(f"\n{'=' * 60}")
    print(f"🚀 开始实验: 负样本采样率 = {neg_rate * 100}%")
    print(f"{'=' * 60}")

    # --- 特征诊断内部函数 ---
    def prune_low_value_features(importance_df, threshold_ratio=0.99):
        print("\n🔍 开始执行 [低价值特征] 诊断...")
        zero_gain_features = importance_df[importance_df["gain"] == 0.0]["column"].tolist()
        
        gain_sum = importance_df["gain"].sum()
        if gain_sum > 0:
            importance_df["gain_ratio"] = importance_df["gain"] / gain_sum
            importance_df["cum_gain_ratio"] = importance_df["gain_ratio"].cumsum()
            tail_features = importance_df[importance_df["cum_gain_ratio"] > threshold_ratio]["column"].tolist()
        else:
            tail_features = []

        features_to_drop = list(set(zero_gain_features + tail_features))
        print(f"   -> 零增益特征数: {len(zero_gain_features)}")
        print(f"   -> 累积贡献末尾(>{threshold_ratio * 100}%)的特征数: {len(tail_features)}")
        return features_to_drop

    # 【核心修正】恢复高相关性特征的过滤，解除共线性干扰
    def find_highly_correlated_features(df_sample, feature_cols, threshold=0.95):
        print(f"\n🔍 开始执行 [高相关性冗余特征] 诊断 (阈值: {threshold})...")
        numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_sample[c])]
        if not numeric_feature_cols:
            return []
        
        sample_df = df_sample.sample(n=min(100000, len(df_sample)), random_state=42)[numeric_feature_cols]
        corr_matrix = sample_df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
        print(f"   -> 发现 {len(to_drop)} 个高冗余特征。")
        return to_drop

    # Step 0: 动态加载黑名单
    if os.path.exists(DROP_FILE_NAME):
        try:
            drop_df = pd.read_csv(DROP_FILE_NAME)
            features_to_drop = drop_df.iloc[:, 0].dropna().tolist()
            local_ignore_cols.extend(features_to_drop)
            local_ignore_cols = list(set(local_ignore_cols))
            print(f"♻️ 成功加载并添加 {len(features_to_drop)} 个待剔除特征。")
        except Exception as e:
            print(f"⚠️ 读取 {DROP_FILE_NAME} 失败: {e}")

    # Step 1: 加载训练集
    print("\n[Step 1] 加载训练集...")
    df_train = load_and_downsample(TRAIN_PATH, neg_sample_rate=neg_rate)
    train_features = [c for c in df_train.columns if c not in local_ignore_cols]
    
    # 【核心修复】：显式拉回 item_category 并保存其绝对字典
    cat_dtype = None # 初始化
    if 'item_category' in df_train.columns:
        if 'item_category' not in train_features:
            train_features.append('item_category')
        df_train['item_category'] = df_train['item_category'].astype('category')
        cat_dtype = df_train['item_category'].dtype # ✅ 提取训练集的基准字典
        
    print(f"最终输入 LightGBM 的特征维度: {len(train_features)}")

    # 【核心修正】移除了 params=LGB_PARAMS，恢复默认 255 分桶上限
    dtrain = lgb.Dataset(df_train[train_features], label=df_train["label"], free_raw_data=True)
    dtrain.construct()
    del df_train
    gc.collect()

    # Step 2: 加载验证集并计算相关性
    print("\n[Step 2] 加载验证集...")
    df_val = load_and_downsample(VAL_PATH, neg_sample_rate=neg_rate)

    high_corr_drops = find_highly_correlated_features(df_val, train_features, threshold=0.95)

    # 【核心修复】：强制套用训练集的基准字典
    if 'item_category' in df_val.columns and cat_dtype is not None:
        df_val['item_category'] = df_val['item_category'].astype(cat_dtype)

    dval = lgb.Dataset(df_val[train_features], label=df_val["label"], reference=dtrain, free_raw_data=True)
    dval.construct()
    del df_val
    gc.collect()
    # Step 3: 训练模型
    print("\n[Step 3] 训练 LightGBM...")
    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
    )

    print(f"\n🏆 [Rate={neg_rate}] Top 15 重要特征:")
    importance = pd.DataFrame({
        "column": model.feature_name(),
        "gain": model.feature_importance(importance_type="gain"),
    }).sort_values(by="gain", ascending=False)
    print(importance.head(15))

    low_value_drops = prune_low_value_features(importance.copy(), threshold_ratio=0.99)
    final_drop_list = list(set(low_value_drops + high_corr_drops))

    print(f"\n💡 综合诊断完毕。建议下一次迭代中剔除以下 {len(final_drop_list)} 个特征:")
    print(final_drop_list)

    pd.Series(final_drop_list, name="drop_features").to_csv(DROP_FILE_NAME, index=False)

    # Step 4: 预测测试集
    print(f"\n[Step 4] 预测测试集 (Top-K={SUBMIT_K})...")
    train_cols = model.feature_name()
    needed_cols = list(set(["user_id", "item_id"] + train_cols))
    parquet_file = pq.ParquetFile(TEST_PATH)

    master_top_k = pd.DataFrame()

    for batch in parquet_file.iter_batches(batch_size=500000, columns=needed_cols):
        df_chunk = batch.to_pandas()
        
        # 【核心修复】：测试集同样需要强制套用类别字典
        if 'item_category' in df_chunk.columns and cat_dtype is not None:
            df_chunk['item_category'] = df_chunk['item_category'].astype(cat_dtype)
            
        X_chunk = df_chunk[train_cols]
        preds = model.predict(X_chunk, num_iteration=model.best_iteration)

        
        chunk_res = pd.DataFrame({
            "user_id": df_chunk["user_id"].values,
            "item_id": df_chunk["item_id"].values,
            "pred_score": preds,
        })

        chunk_res = chunk_res.sort_values("pred_score", ascending=False).head(SUBMIT_K)
        master_top_k = pd.concat([master_top_k, chunk_res], ignore_index=True)
        master_top_k = master_top_k.sort_values("pred_score", ascending=False).head(SUBMIT_K)

        del df_chunk, X_chunk, preds, chunk_res
        gc.collect()

    filename = f"tianchi_predict_neg_rate_{neg_rate}_v3.txt"
    save_path = os.path.join(OUTPUT_DIR, filename)
    master_top_k[["user_id", "item_id"]].to_csv(save_path, sep="\t", index=False)

    print(f"\n🎉 实验完成，结果已保存: {save_path}")
    del dtrain, dval, model, master_top_k

run_ablation_experiment(neg_rate=0.05, ignore_cols=IGNORE_COLS)


🚀 开始实验: 负样本采样率 = 5.0%

[Step 1] 加载训练集...
  -> 读取文件: train_set_1217_2026-0331.parquet | 采样率: 0.05
     ✅ 加载完成. 最终行数: 914048
最终输入 LightGBM 的特征维度: 337
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.

[Step 2] 加载验证集...
  -> 读取文件: val_set_1218_2026-0331.parquet | 采样率: 0.05
     ✅ 加载完成. 最终行数: 911132

🔍 开始执行 [高相关性冗余特征] 诊断 (阈值: 0.95)...
   -> 发现 176 个高冗余特征。

[Step 3] 训练 LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.959053	valid_1's auc: 0.943236
[100]	training's auc: 0.969424	valid_1's auc: 0.946748
[150]	training's auc: 0.974998	valid_1's auc: 0.947673
[200]	training's auc: 0.978763	valid_1's auc: 0.948031
[250]	training's auc: 0.981751	valid_1's auc: 0.948102
[300]	training's auc: 0.984504	valid_1's auc: 0.948077
Early stopping, best iteration is:
[276]	training

In [11]:
import polars as pl
import glob
from datetime import datetime

def evaluate_recall_coverage(val_parquet_path: str, raw_user_files: list, target_date_str: str, item_set: set):
    """
    评估验证集上的多路召回真实覆盖率 (修正版)
    """
    print(f"📊 开始评估 {target_date_str} 的多路召回真实覆盖率...\n")

    # ==========================================
    # 1. 获取分子：从 Parquet 验证集中统计命中情况
    # ==========================================
    # 仅加载需要的列以节省内存
    df_val = pl.read_parquet(
        val_parquet_path, 
        columns=["user_id", "item_id", "label", "recall_u2i", "recall_itemcf", "recall_global"]
    )
    
    # 过滤出成功被召回且真实发生购买的样本 (label == 1)
    df_hits = df_val.filter(pl.col("label") == 1)
    
    # 统计各路分别命中的数量
    hit_u2i = df_hits.filter(pl.col("recall_u2i") == 1).height
    hit_itemcf = df_hits.filter(pl.col("recall_itemcf") == 1).height
    hit_global = df_hits.filter(pl.col("recall_global") == 1).height

    # 统计总去重命中数 (任一策略召回成功即算命中)
    hit_any = df_hits.filter(
        (pl.col("recall_u2i") == 1) | 
        (pl.col("recall_itemcf") == 1) | 
        (pl.col("recall_global") == 1)
    ).height

    # ==========================================
    # 2. 获取分母：从原始日志中统计当天的全局真实购买量
    # ==========================================
    lf_raw = pl.scan_parquet(raw_user_files)
    
    # 全局真实购买行为（仅限在官方候选商品池内的购买）
    df_ground_truth = (
        lf_raw.filter(
            (pl.col("time").str.starts_with(target_date_str)) & 
            (pl.col("behavior_type") == 4) &
            (pl.col("item_id").is_in(list(item_set)))
        )
        .select(["user_id", "item_id"])
        .unique()
        .collect()
    )
    total_actual_buys = df_ground_truth.height

    # ==========================================
    # 3. 输出评估指标
    # ==========================================
    global_recall = (hit_any / total_actual_buys) * 100 if total_actual_buys > 0 else 0
    
    print(f"🎯 【全局真实评估】")
    print(f" -> 目标日期 ({target_date_str}) 官方商品池内真实购买总数: {total_actual_buys}")
    print(f" -> 多路召回成功捕获的去重购买数: {hit_any}")
    print(f" -> 综合全局召回率 (Global Recall): {global_recall:.2f}%\n")
    
    print(f"🧩 【单路召回命中贡献 (含重叠交叉)】")
    print(f" -> U2I 召回命中: {hit_u2i} 个")
    print(f" -> ItemCF 召回命中: {hit_itemcf} 个")
    print(f" -> Global 召回命中: {hit_global} 个\n")

    return hit_any, total_actual_buys

# ==========================================
# 执行示例
# ==========================================
val_file = "./val_set_1218_2026-0331.parquet"
user_files = sorted(glob.glob("Dataset/train_user_*.parquet"))
item_files = sorted(glob.glob("Dataset/train_item_*.parquet"))

# 重建 item_set
item_set = set()
for f in item_files:
    item_set.update(pl.read_parquet(f, columns=["item_id"])["item_id"].unique().to_list())

evaluate_recall_coverage(
    val_parquet_path=val_file, 
    raw_user_files=user_files, 
    target_date_str="2014-12-18", 
    item_set=item_set
)

📊 开始评估 2014-12-18 的多路召回真实覆盖率...

🎯 【全局真实评估】
 -> 目标日期 (2014-12-18) 官方商品池内真实购买总数: 33701
 -> 多路召回成功捕获的去重购买数: 9574
 -> 综合全局召回率 (Global Recall): 28.41%

🧩 【单路召回命中贡献 (含重叠交叉)】
 -> U2I 召回命中: 9320 个
 -> ItemCF 召回命中: 3359 个
 -> Global 召回命中: 44 个



(9574, 33701)

In [12]:
import polars as pl

def analyze_recall_redundancy(val_parquet_path: str):
    print("🔍 开始深入分析多路召回冗余度与真实增量贡献...\n")

    # 使用 lazy 模式降低内存占用
    lf_val = pl.scan_parquet(val_parquet_path)
    
    # 定义当前的召回标识列
    recall_cols = ["recall_u2i", "recall_itemcf", "recall_global"]
    
    # ==========================================
    # 1. 真实购买样本 (Label=1) 的召回重叠情况 (核心！看谁在真正起作用)
    # ==========================================
    print("🎯 1. 真实购买样本 (Label=1) 的召回重叠情况 (核心增量评估):")
    hits_overlap = (
        lf_val.filter(pl.col("label") == 1)
        .group_by(recall_cols)
        .agg(pl.len().alias("count"))
        .with_columns((pl.col("count") / pl.col("count").sum() * 100).round(2).alias("ratio(%)"))
        .sort("count", descending=True)
        .collect()
    )
    print(hits_overlap)
    
    # ==========================================
    # 2. 整体候选集的召回重叠情况 (看谁在疯狂制造算力负担)
    # ==========================================
    print("\n📦 2. 整体候选集的召回重叠情况 (算力冗余与噪音评估):")
    cand_overlap = (
        lf_val
        .group_by(recall_cols)
        .agg(pl.len().alias("count"))
        .with_columns((pl.col("count") / pl.col("count").sum() * 100).round(2).alias("ratio(%)"))
        .sort("count", descending=True)
        .collect()
    )
    print(cand_overlap)

    # ==========================================
    # 3. 提取独家召回增量 (Exclusive Recall)
    # ==========================================
    print("\n💡 【独家命中统计 (仅被单一策略成功抓取的真实购买)】")
    for col in recall_cols:
        # 筛选条件：当前策略为1，其他策略为0
        other_cols = [c for c in recall_cols if c != col]
        exclusive_hits = hits_overlap.filter(
            (pl.col(col) == 1) & 
            (pl.col(other_cols[0]) == 0) & 
            (pl.col(other_cols[1]) == 0)
        )
        
        if exclusive_hits.height > 0:
            count = exclusive_hits["count"][0]
            ratio = exclusive_hits["ratio(%)"][0]
            print(f" -> {col} 独家增量: {count} 个购买样本 (占命中总数 {ratio}%)")
        else:
            print(f" -> {col} 独家增量: 0 个 (该策略毫无独立贡献！)")

# 运行验证集数据
val_file = "./val_set_1218_2026-0331.parquet"
analyze_recall_redundancy(val_file)

🔍 开始深入分析多路召回冗余度与真实增量贡献...

🎯 1. 真实购买样本 (Label=1) 的召回重叠情况 (核心增量评估):
shape: (5, 5)
┌────────────┬───────────────┬───────────────┬───────┬──────────┐
│ recall_u2i ┆ recall_itemcf ┆ recall_global ┆ count ┆ ratio(%) │
│ ---        ┆ ---           ┆ ---           ┆ ---   ┆ ---      │
│ i8         ┆ i8            ┆ i8            ┆ u32   ┆ f64      │
╞════════════╪═══════════════╪═══════════════╪═══════╪══════════╡
│ 1          ┆ 0             ┆ 0             ┆ 6171  ┆ 64.46    │
│ 1          ┆ 1             ┆ 0             ┆ 3143  ┆ 32.83    │
│ 0          ┆ 1             ┆ 0             ┆ 216   ┆ 2.26     │
│ 0          ┆ 0             ┆ 1             ┆ 38    ┆ 0.4      │
│ 1          ┆ 0             ┆ 1             ┆ 6     ┆ 0.06     │
└────────────┴───────────────┴───────────────┴───────┴──────────┘

📦 2. 整体候选集的召回重叠情况 (算力冗余与噪音评估):
shape: (5, 5)
┌────────────┬───────────────┬───────────────┬─────────┬──────────┐
│ recall_u2i ┆ recall_itemcf ┆ recall_global ┆ count   ┆ ratio(%) │
│ ---      